In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv("smartcart_customers.csv")

In [3]:
data.sample(5)

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,Complain,Response
1085,6295,1977,Graduation,Married,54233.0,0,1,04-02-2013,46,652,...,21,0,142,4,11,6,5,8,0,0
2215,692,1954,Graduation,Together,36807.0,1,1,20-12-2012,88,4,...,0,0,5,1,1,0,2,6,0,0
897,5046,1958,Graduation,Divorced,78331.0,0,1,23-01-2014,97,756,...,160,169,61,2,10,6,7,4,0,0
377,4459,1989,Graduation,Single,30279.0,1,0,30-12-2012,13,10,...,4,4,1,1,1,0,3,8,0,0
631,9624,1958,PhD,Together,65488.0,0,0,13-06-2014,46,603,...,36,18,54,1,3,4,6,1,0,0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

In [5]:
data["Year_Birth"].dtype

dtype('int64')

In [6]:
data.isnull().sum()

ID                      0
Year_Birth              0
Education               0
Marital_Status          0
Income                 24
Kidhome                 0
Teenhome                0
Dt_Customer             0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
Complain                0
Response                0
dtype: int64

# Data Preprocessing

## 1. Handle Missing Values

In [7]:
data["Income"] = data["Income"].fillna(data["Income"].median())

In [8]:
data.isnull().sum()

ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 0
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
Complain               0
Response               0
dtype: int64

## 2. Feature Engineering

In [9]:
# Birth-Date -> Age
data["Age"] = 2026-data["Year_Birth"]

In [10]:
data.sample()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,Complain,Response,Age
1210,5644,1957,Graduation,Together,42213.0,0,1,12-10-2013,96,309,...,3,20,2,5,1,7,7,0,0,69


In [11]:
# Customer-Joining-Date -> Customer Tenure Days

data["Dt_Customer"] = pd.to_datetime(data["Dt_Customer"],dayfirst=True)

reference_date = data["Dt_Customer"].max()

data["Customer_Tenure_Days"] = (reference_date - data["Dt_Customer"]).dt.days

In [12]:
# Total Spending :

data["Total_Spending"] = (data["MntWines"] + data["MntFruits"] + data["MntMeatProducts"] 
                        + data["MntFishProducts"] + data["MntSweetProducts"]+ data["MntGoldProds"])

In [13]:
# Total Children :

data["Total_Children"] = (data["Kidhome"] + data["Teenhome"])

In [14]:
# Education :

data["Education"].value_counts()

data["Education"] = data["Education"].replace({
    "Graduation" : "Graduate",
    "PhD" : "Postgraduate","Master" : "Postgraduate",
    "2n Cycle" : "Undergraduate","Basic" : "Undergraduate"
})

In [15]:
# Marital-Status :

data["Marital_Status"].value_counts()

data["Living_With"] = data["Marital_Status"].replace({
    "Married" : "Partner","Together" : "Partner",
    "Divorced" : "Single","Widow" : "Single","Alone" : "Single","Absurd" : "Single","YOLO" : "Single"
})

## 3. Drop Columns

In [16]:
data.columns

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'Complain', 'Response', 'Age', 'Customer_Tenure_Days', 'Total_Spending',
       'Total_Children', 'Living_With'],
      dtype='object')

In [17]:
drop_cols = ["ID","Year_Birth","Marital_Status","Kidhome","Teenhome",
             "MntMeatProducts","Dt_Customer","MntWines","MntFruits","MntMeatProducts",
             "MntFishProducts","MntSweetProducts","MntGoldProds"]

data_cleaned = data.drop(drop_cols,axis=1)

In [18]:
data_cleaned.shape

(2240, 15)

## 4. Outliers

In [ ]:
cols = ["Income","Recency","Response","Age","Total_Spending","Total_Children"]

# Relative plots of some features - (Pair Plots)

sns.pairplot(data_cleaned[cols])

In [ ]:
# Remove the outliers :

print("Data size with outliers : ",len(data_cleaned))

data_cleaned = data_cleaned[(data_cleaned["Age"] < 90)]
data_cleaned = data_cleaned[(data_cleaned["Income"] < 600000)]

print("Data size without outliers : ",len(data_cleaned))